# Comprehensive Build Trace Analysis Guide

This notebook demonstrates how to analyze C++ build performance using Clang's `-ftime-trace` feature. We'll explore the trace analysis library and show practical techniques for understanding and improving compilation times.

## The Problem: C++ Metaprogramming Build Times

The Composable Kernel (CK) library uses extensive C++17 metaprogramming to generate high-performance GPU kernels. While this approach provides excellent runtime performance, it comes with a cost: **long compilation times**.

Understanding where the compiler spends its time is critical for:
- **Identifying bottlenecks**: Which templates are most expensive?
- **Measuring progress**: Are our optimizations working?
- **Focusing efforts**: Where should we invest time to improve build performance?

## The Solution: Data-Driven Analysis

Clang's `-ftime-trace` flag generates detailed JSON files showing exactly where compilation time is spent. This notebook shows how to:

1. **Parse** trace files efficiently using parallel processing
2. **Transform** raw JSON into analyzable pandas DataFrames
3. **Analyze** build performance with practical examples
4. **Visualize** results to communicate findings

Let's treat this as a **big data problem** and use the best tools available: pandas, parallel processing, and Jupyter notebooks.

## Setup and Imports

First, let's import the necessary libraries and set up our environment.

In [ ]:
import sys
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
import time

import pandas as pd

# Add parent directory to path to import trace_analysis
sys.path.insert(0, str(Path.cwd().parent))

from trace_analysis import TraceFile, TraceParser, TraceTransformer

# Optional: tqdm for progress bars
try:
    from tqdm.auto import tqdm

    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("Note: Install tqdm for progress bars: pip install tqdm")

# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)

print(f"Using {cpu_count()} CPU cores for parallel processing")
print(f"Pandas version: {pd.__version__}")

## Part 1: Single File Analysis

Let's start by analyzing a single trace file to understand the data structure and basic analysis patterns.

### Understanding the Trace File Format

Clang's `-ftime-trace` generates JSON files in the [Chrome Trace Event Format](https://docs.google.com/document/d/1CvAClvFfyA5R-PhYUmn5OOQtYMH4h6I0nSsKchNAySU/preview). Each file contains:

- **traceEvents**: Array of compilation events (parsing, template instantiation, code generation, etc.)
- **beginningOfTime**: Timestamp reference point

Each event has:
- `name`: Event type (e.g., "InstantiateFunction", "ParseClass")
- `dur`: Duration in microseconds
- `ts`: Timestamp in microseconds
- `args.detail`: Additional information (e.g., template signature)

In [ ]:
# Configure the path to your trace files
# Adjust this path to point to your build-trace directory
TRACE_DIR = Path("../../../build-trace")

# Find a sample trace file
sample_files = list(TRACE_DIR.rglob("*.json"))

if not sample_files:
    print(f"No trace files found in {TRACE_DIR}")
    print("\nTo generate trace files:")
    print("1. Configure your build with: cmake -DCMAKE_CXX_FLAGS='-ftime-trace' ...")
    print("2. Build your project")
    print("3. Trace files will be generated alongside object files")
else:
    print(f"Found {len(sample_files):,} trace files")
    sample_file = sample_files[0]
    print(f"\nUsing sample file: {sample_file.name}")
    print(f"File size: {sample_file.stat().st_size / 1024:.1f} KB")

### Parsing a Single File

The trace_analysis library provides a simple three-step process:

1. **TraceFile**: Create a metadata wrapper
2. **TraceParser**: Parse JSON into Python dictionaries
3. **TraceTransformer**: Convert to pandas DataFrames

In [ ]:
if sample_files:
    # Step 1: Create TraceFile metadata wrapper
    trace_file = TraceFile.from_path(sample_file)
    print(f"Trace file: {trace_file.path.name}")
    print(f"Size: {trace_file.size_bytes:,} bytes")

    # Step 2: Parse JSON to Python dictionaries
    start = time.time()
    events = TraceParser.parse(trace_file)
    parse_time = time.time() - start
    print(f"\nParsed {len(events):,} events in {parse_time:.3f}s")

    # Step 3: Convert to DataFrames
    start = time.time()
    events_df = TraceTransformer.to_events_dataframe(events)
    templates_df = TraceTransformer.to_templates_dataframe(events)
    transform_time = time.time() - start
    print(f"Transformed to DataFrames in {transform_time:.3f}s")

    print(f"\nEvents DataFrame: {len(events_df):,} rows")
    print(f"Templates DataFrame: {len(templates_df):,} rows")

### Examining the Events DataFrame

The events DataFrame contains all compilation events with optimized data types for memory efficiency.

In [ ]:
if sample_files:
    print("Events DataFrame Schema:")
    print(events_df.dtypes)
    print("\nFirst few events:")
    display(events_df.head(10))

    print("\nMemory usage:")
    print(events_df.memory_usage(deep=True))

### Basic Statistics for a Single File

In [ ]:
if sample_files:
    total_duration_us = events_df["dur"].sum()
    total_duration_s = total_duration_us / 1e6

    print(f"Total compilation time: {total_duration_s:.2f} seconds")
    print(f"Total events: {len(events_df):,}")
    print(f"Average event duration: {events_df['dur'].mean() / 1e3:.2f} ms")
    print(f"Median event duration: {events_df['dur'].median() / 1e3:.2f} ms")
    print(f"Max event duration: {events_df['dur'].max() / 1e6:.2f} s")

### Top Event Types

Let's see which types of compilation events take the most time.

In [ ]:
if sample_files:
    # Aggregate by event type
    event_stats = (
        events_df.groupby("name", observed=True)["dur"]
        .agg(
            [
                ("count", "count"),
                ("total_us", "sum"),
                ("mean_us", "mean"),
                ("median_us", "median"),
                ("max_us", "max"),
            ]
        )
        .sort_values("total_us", ascending=False)
    )

    # Convert to more readable units
    event_stats["total_s"] = event_stats["total_us"] / 1e6
    event_stats["mean_ms"] = event_stats["mean_us"] / 1e3
    event_stats["median_ms"] = event_stats["median_us"] / 1e3
    event_stats["max_ms"] = event_stats["max_us"] / 1e3
    event_stats["pct_total"] = (event_stats["total_us"] / total_duration_us) * 100

    print("Top 15 Event Types by Total Duration:")
    display(
        event_stats[["count", "total_s", "mean_ms", "max_ms", "pct_total"]].head(15)
    )

### Template Instantiation Analysis

Template instantiation is often the biggest contributor to C++ metaprogramming build times. Let's analyze it in detail.

In [ ]:
if sample_files and len(templates_df) > 0:
    template_time_us = templates_df["dur"].sum()
    template_time_s = template_time_us / 1e6
    template_pct = (template_time_us / total_duration_us) * 100

    print("Template Instantiation Summary:")
    print(f"  Total instantiations: {len(templates_df):,}")
    print(f"  Total time: {template_time_s:.2f}s")
    print(f"  Percentage of build time: {template_pct:.1f}%")
    print(f"  Average per instantiation: {templates_df['dur'].mean() / 1e3:.2f} ms")
    print(f"  Median per instantiation: {templates_df['dur'].median() / 1e3:.2f} ms")

In [ ]:
if sample_files and len(templates_df) > 0:
    # Most expensive individual template instantiations
    print("Top 10 Slowest Individual Template Instantiations:")
    slowest = templates_df.nlargest(10, "dur")[
        ["name", "dur", "template_detail"]
    ].copy()
    slowest["dur_ms"] = slowest["dur"] / 1e3
    display(slowest[["name", "dur_ms", "template_detail"]])

In [ ]:
if sample_files and len(templates_df) > 0:
    # Most frequently instantiated templates
    print("Top 15 Most Frequently Instantiated Templates:")
    template_counts = templates_df["template_detail"].value_counts().head(15)
    display(
        pd.DataFrame(
            {"template": template_counts.index, "count": template_counts.values}
        )
    )

In [ ]:
if sample_files and len(templates_df) > 0:
    # Most expensive templates by total time
    print("Top 15 Most Expensive Templates by Total Duration:")
    template_totals = (
        templates_df.groupby("template_detail")["dur"]
        .agg(
            [
                ("count", "count"),
                ("total_us", "sum"),
                ("mean_us", "mean"),
                ("max_us", "max"),
            ]
        )
        .sort_values("total_us", ascending=False)
    )

    template_totals["total_s"] = template_totals["total_us"] / 1e6
    template_totals["mean_ms"] = template_totals["mean_us"] / 1e3
    template_totals["max_ms"] = template_totals["max_us"] / 1e3

    display(template_totals[["count", "total_s", "mean_ms", "max_ms"]].head(15))

## Part 2: Multi-File Analysis

Now let's scale up to analyze an entire build. We'll use parallel processing to handle thousands of trace files efficiently.

### Parallel Processing Strategy

The key insight: **parsing is I/O bound**, so we can process multiple files in parallel using all CPU cores. The library uses:

- `ProcessPoolExecutor` for true parallelism (not limited by Python's GIL)
- `orjson` for fast JSON parsing (1.65x faster than stdlib)
- Optimized pandas dtypes to minimize memory usage

**Performance**: On a typical build with 4,484 trace files (~46 GB), we can parse everything in ~26 seconds.

In [ ]:
def process_file(json_path: Path) -> tuple:
    """
    Process a single trace file and return DataFrames.

    This function is designed to be called in parallel by ProcessPoolExecutor.
    """
    trace_file = TraceFile.from_path(json_path)
    events = TraceParser.parse(trace_file)
    events_df = TraceTransformer.to_events_dataframe(events)
    templates_df = TraceTransformer.to_templates_dataframe(events)

    return (
        str(json_path.name),
        events_df,
        templates_df,
    )


print("Parallel processing function defined")

In [ ]:
# Find all trace files
json_files = list(TRACE_DIR.rglob("*.json"))

if not json_files:
    print(f"No trace files found in {TRACE_DIR}")
else:
    print(f"Found {len(json_files):,} trace files")

    # For demonstration, you might want to limit the number of files
    # Uncomment the next line to process only the first 100 files
    # json_files = json_files[:100]

    total_size = sum(f.stat().st_size for f in json_files)
    print(f"Total size: {total_size / 1024**3:.2f} GB")

### Processing All Files in Parallel

This cell will process all trace files using all available CPU cores. Depending on the number of files and your system, this may take a few minutes.

In [ ]:
if json_files:
    print(f"Processing {len(json_files):,} files with {cpu_count()} workers...\n")

    start_time = time.time()
    all_events = []
    all_templates = []
    file_names = []

    # Submit all files for parallel processing
    with ProcessPoolExecutor(max_workers=cpu_count()) as executor:
        futures = {executor.submit(process_file, f): f for f in json_files}

        # Collect results with progress bar
        if HAS_TQDM:
            pbar = tqdm(total=len(json_files), desc="Processing", unit="files")

        for future in as_completed(futures):
            file_name, events_df, templates_df = future.result()

            file_names.append(file_name)
            all_events.append(events_df)
            all_templates.append(templates_df)

            if HAS_TQDM:
                pbar.update(1)

        if HAS_TQDM:
            pbar.close()

    parse_time = time.time() - start_time
    print(
        f"\nParsing complete in {parse_time:.2f}s ({len(json_files) / parse_time:.1f} files/sec)"
    )

    # Combine all DataFrames
    print("Combining results...")
    combine_start = time.time()

    # Filter out empty DataFrames
    non_empty_events = [df for df in all_events if len(df) > 0]
    non_empty_templates = [df for df in all_templates if len(df) > 0]

    all_events_df = (
        pd.concat(non_empty_events, ignore_index=True)
        if non_empty_events
        else pd.DataFrame()
    )
    all_templates_df = (
        pd.concat(non_empty_templates, ignore_index=True)
        if non_empty_templates
        else pd.DataFrame()
    )

    combine_time = time.time() - combine_start
    total_time = time.time() - start_time

    print(f"Combined in {combine_time:.2f}s")
    print(f"\nTotal analysis time: {total_time:.2f}s")
    print("\nCombined DataFrames:")
    print(f"  Events: {len(all_events_df):,} rows")
    print(f"  Templates: {len(all_templates_df):,} rows")
    print(
        f"  Memory usage: {(all_events_df.memory_usage(deep=True).sum() + all_templates_df.memory_usage(deep=True).sum()) / 1024**3:.2f} GB"
    )

### Build-Wide Statistics

Now we have all compilation events in a single DataFrame. Let's analyze the entire build.

In [ ]:
if json_files and len(all_events_df) > 0:
    total_build_time_us = all_events_df["dur"].sum()
    total_build_time_s = total_build_time_us / 1e6
    total_build_time_min = total_build_time_s / 60

    print("=" * 80)
    print("BUILD-WIDE STATISTICS")
    print("=" * 80)
    print(f"Files processed: {len(json_files):,}")
    print(f"Total events: {len(all_events_df):,}")
    print(
        f"Total build time: {total_build_time_min:.2f} minutes ({total_build_time_s:.2f} seconds)"
    )
    print(f"Average time per file: {total_build_time_s / len(json_files):.2f} seconds")
    print("=" * 80)

### Top Event Types Across Entire Build

In [ ]:
if json_files and len(all_events_df) > 0:
    print("Top 20 Event Types by Total Duration Across All Files:")

    build_event_stats = (
        all_events_df.groupby("name", observed=True)["dur"]
        .agg(
            [
                ("count", "count"),
                ("total_us", "sum"),
                ("mean_us", "mean"),
                ("max_us", "max"),
            ]
        )
        .sort_values("total_us", ascending=False)
    )

    build_event_stats["total_s"] = build_event_stats["total_us"] / 1e6
    build_event_stats["total_min"] = build_event_stats["total_s"] / 60
    build_event_stats["mean_ms"] = build_event_stats["mean_us"] / 1e3
    build_event_stats["max_ms"] = build_event_stats["max_us"] / 1e3
    build_event_stats["pct_total"] = (
        build_event_stats["total_us"] / total_build_time_us
    ) * 100

    display(
        build_event_stats[
            ["count", "total_min", "mean_ms", "max_ms", "pct_total"]
        ].head(20)
    )

### Build-Wide Template Analysis

This is where we can identify the most expensive templates across the entire build.

In [ ]:
if json_files and len(all_templates_df) > 0:
    total_template_time_us = all_templates_df["dur"].sum()
    total_template_time_s = total_template_time_us / 1e6
    total_template_time_min = total_template_time_s / 60
    template_pct = (total_template_time_us / total_build_time_us) * 100

    print("=" * 80)
    print("BUILD-WIDE TEMPLATE INSTANTIATION SUMMARY")
    print("=" * 80)
    print(f"Total template instantiations: {len(all_templates_df):,}")
    print(
        f"Total template time: {total_template_time_min:.2f} minutes ({total_template_time_s:.2f} seconds)"
    )
    print(f"Percentage of total build time: {template_pct:.1f}%")
    print(f"Average per instantiation: {all_templates_df['dur'].mean() / 1e3:.2f} ms")
    print(f"Median per instantiation: {all_templates_df['dur'].median() / 1e3:.2f} ms")
    print("=" * 80)

In [ ]:
if json_files and len(all_templates_df) > 0:
    print("Top 20 Most Expensive Templates by Total Duration:")

    build_template_stats = (
        all_templates_df.groupby("template_detail")["dur"]
        .agg(
            [
                ("count", "count"),
                ("total_us", "sum"),
                ("mean_us", "mean"),
                ("median_us", "median"),
                ("max_us", "max"),
            ]
        )
        .sort_values("total_us", ascending=False)
    )

    build_template_stats["total_s"] = build_template_stats["total_us"] / 1e6
    build_template_stats["mean_ms"] = build_template_stats["mean_us"] / 1e3
    build_template_stats["median_ms"] = build_template_stats["median_us"] / 1e3
    build_template_stats["max_ms"] = build_template_stats["max_us"] / 1e3
    build_template_stats["pct_template_time"] = (
        build_template_stats["total_us"] / total_template_time_us
    ) * 100

    display(
        build_template_stats[
            ["count", "total_s", "mean_ms", "median_ms", "max_ms", "pct_template_time"]
        ].head(20)
    )

In [ ]:
if json_files and len(all_templates_df) > 0:
    print("Top 20 Most Frequently Instantiated Templates:")

    template_frequency = all_templates_df["template_detail"].value_counts().head(20)
    display(
        pd.DataFrame(
            {
                "template": template_frequency.index,
                "instantiation_count": template_frequency.values,
            }
        )
    )

## Part 3: Advanced Analysis Patterns

Now that we have all the data, let's explore some advanced analysis techniques.

### Pattern 1: Finding Optimization Opportunities

Templates that are both frequently instantiated AND expensive per instantiation are prime optimization targets.

In [ ]:
if json_files and len(all_templates_df) > 0:
    # Calculate a "priority score" for optimization
    # High count + high average time = high priority

    optimization_targets = all_templates_df.groupby("template_detail")["dur"].agg(
        [("count", "count"), ("total_us", "sum"), ("mean_us", "mean")]
    )

    # Normalize count and mean to 0-1 range
    optimization_targets["count_norm"] = (
        optimization_targets["count"] - optimization_targets["count"].min()
    ) / (optimization_targets["count"].max() - optimization_targets["count"].min())
    optimization_targets["mean_norm"] = (
        optimization_targets["mean_us"] - optimization_targets["mean_us"].min()
    ) / (optimization_targets["mean_us"].max() - optimization_targets["mean_us"].min())

    # Priority score: weighted combination of frequency and cost
    optimization_targets["priority_score"] = (
        0.5 * optimization_targets["count_norm"]
        + 0.5 * optimization_targets["mean_norm"]
    )

    optimization_targets["total_s"] = optimization_targets["total_us"] / 1e6
    optimization_targets["mean_ms"] = optimization_targets["mean_us"] / 1e3

    print("Top 15 Optimization Targets (High Frequency + High Cost):")
    display(
        optimization_targets.sort_values("priority_score", ascending=False)[
            ["count", "total_s", "mean_ms", "priority_score"]
        ].head(15)
    )

### Pattern 2: Template Instantiation Distribution

Understanding the distribution of template instantiation times can reveal outliers and patterns.

In [ ]:
if json_files and len(all_templates_df) > 0:
    # Calculate percentiles
    percentiles = [50, 75, 90, 95, 99, 99.9]
    template_percentiles = all_templates_df["dur"].quantile(
        [p / 100 for p in percentiles]
    )

    print("Template Instantiation Time Distribution:")
    print(f"{'Percentile':<15} {'Duration (ms)':>15}")
    print("-" * 30)
    for p, val in zip(percentiles, template_percentiles):
        print(f"P{p:<14} {val / 1e3:>15.2f}")

    # Count templates in different time buckets
    print("\nTemplate Count by Duration Bucket:")
    buckets = [
        (0, 1e3, "< 1ms"),
        (1e3, 10e3, "1-10ms"),
        (10e3, 100e3, "10-100ms"),
        (100e3, 1e6, "100ms-1s"),
        (1e6, float("inf"), "> 1s"),
    ]

    for low, high, label in buckets:
        count = (
            (all_templates_df["dur"] >= low) & (all_templates_df["dur"] < high)
        ).sum()
        pct = (count / len(all_templates_df)) * 100
        print(f"{label:<15} {count:>10,} ({pct:>5.1f}%)")

### Pattern 3: Event Type Breakdown by Template vs Non-Template

Compare template-related events to other compilation activities.

In [ ]:
if json_files and len(all_events_df) > 0:
    # Identify template-related event types
    template_event_types = [
        "InstantiateClass",
        "InstantiateFunction",
        "InstantiateVariable",
        "InstantiateAlias",
    ]

    # Calculate time spent on template vs non-template events
    template_mask = all_events_df["name"].isin(template_event_types)

    template_time = all_events_df[template_mask]["dur"].sum()
    non_template_time = all_events_df[~template_mask]["dur"].sum()

    print("Build Time Breakdown:")
    print(f"{'Category':<30} {'Time (min)':>15} {'Percentage':>12}")
    print("-" * 60)
    print(
        f"{'Template Instantiation':<30} {template_time / 1e6 / 60:>15.2f} {(template_time / total_build_time_us) * 100:>11.1f}%"
    )
    print(
        f"{'Other Compilation':<30} {non_template_time / 1e6 / 60:>15.2f} {(non_template_time / total_build_time_us) * 100:>11.1f}%"
    )
    print("-" * 60)
    print(f"{'Total':<30} {total_build_time_us / 1e6 / 60:>15.2f} {'100.0%':>12}")

### Pattern 4: Identifying Expensive Template Patterns

Look for common patterns in expensive templates (e.g., nested templates, specific type combinations).

In [ ]:
if json_files and len(all_templates_df) > 0:
    # Extract template base names (before '<')
    all_templates_df["template_base"] = (
        all_templates_df["template_detail"].str.split("<").str[0]
    )

    # Analyze by template base name
    base_stats = (
        all_templates_df.groupby("template_base")["dur"]
        .agg([("count", "count"), ("total_us", "sum"), ("mean_us", "mean")])
        .sort_values("total_us", ascending=False)
    )

    base_stats["total_s"] = base_stats["total_us"] / 1e6
    base_stats["mean_ms"] = base_stats["mean_us"] / 1e3

    print("Top 20 Template Base Names by Total Time:")
    display(base_stats[["count", "total_s", "mean_ms"]].head(20))

## Part 4: Practical Recommendations

Based on the analysis above, here are practical steps for improving build times:

### 1. Focus on High-Impact Templates

The templates identified in the "Optimization Targets" section should be your first priority. Consider:
- **Explicit instantiation**: Pre-instantiate common template combinations
- **Extern templates**: Declare templates as extern to avoid redundant instantiations
- **Simplification**: Can the template be simplified or split into smaller pieces?

### 2. Measure Progress

After making changes:
1. Rebuild with `-ftime-trace`
2. Re-run this analysis
3. Compare the "Top Templates" tables to see if your changes helped

### 3. Use This Analysis Regularly

Make this analysis part of your development workflow:
- Run it weekly to track build time trends
- Use it to review pull requests that add new templates
- Share results with the team to raise awareness

### 4. Consider Build System Optimizations

Beyond code changes:
- Use precompiled headers for common includes
- Enable unity builds for related source files
- Use ccache or similar tools for incremental builds
- Consider distributed compilation (distcc, icecc)

## Conclusion

This notebook demonstrated how to:

1. **Parse** Clang `-ftime-trace` files efficiently using parallel processing
2. **Transform** raw JSON into analyzable pandas DataFrames
3. **Analyze** build performance at both file and build-wide levels
4. **Identify** optimization opportunities using data-driven techniques

The trace_analysis library makes it easy to treat C++ build performance as a big data problem, using the best tools available: pandas, parallel processing, and Jupyter notebooks.

### Next Steps

- Customize this notebook for your specific analysis needs
- Add visualizations using matplotlib or plotly
- Create automated reports for CI/CD pipelines
- Share insights with your team to drive build time improvements

### Resources

- [Clang -ftime-trace Documentation](https://releases.llvm.org/11.0.0/tools/clang/docs/ClangCommandLineReference.html#cmdoption-clang-ftime-trace)
- [Chrome Trace Event Format](https://docs.google.com/document/d/1CvAClvFfyA5R-PhYUmn5OOQtYMH4h6I0nSsKchNAySU/preview)
- [Template Metaprogramming Performance Talk](https://www.youtube.com/watch?v=vwrXHznaYLA)
- [trace_analysis Library Documentation](../trace_analysis/README.md)